# Chẩn đoán: vì sao chuyên gia CIC-UNSW-NB15 chỉ đạt Macro-F1 khoảng 0,67?

**Bối cảnh.** Ở bản NetFlow (định dạng NetFlow phiên bản 3), chuyên gia UNSW-NB15 đạt Macro-F1 = 1,000 trong miền. Sang bản CICFlowMeter (công cụ trích đặc trưng luồng của Viện An ninh mạng Canada), chuyên gia MLP (Multilayer Perceptron – mạng nơ-ron nhiều lớp) đạt 0,85–0,90, nhưng chuyên gia LightGBM (Light Gradient Boosting Machine – cây tăng cường gradient) chỉ đạt 0,51–0,67 và dao động mạnh giữa các lần chạy. Notebook này kiểm tra 4 giả thuyết:

| Giả thuyết | Cách kiểm tra |
|---|---|
| **G1.** Mô hình vẫn xếp hạng tốt, chỉ là **ngưỡng 0,5** chưa hợp với tỷ lệ tấn công rất thấp (2,5%) | So Macro-F1 ở ngưỡng 0,5 với Macro-F1 ở ngưỡng tốt nhất; xem AUROC (diện tích dưới đường cong ROC – Receiver Operating Characteristic) |
| **G2.** **Nhãn bị "lẫn"**: nhiều luồng mang nhãn tấn công nhưng có đặc trưng GIỐNG HỆT một luồng bình thường, nên không mô hình nào tách được | Đếm số luồng tấn công có "bản sao bình thường" (cùng toàn bộ đặc trưng), theo từng loại tấn công |
| **G4.** LightGBM bị "hỏng" do **trọng số lớp tấn công quá lớn** (khoảng 39 lần, vì tấn công chỉ chiếm 2,5%) và không có dừng sớm | So 5 cấu hình LightGBM: như hiện tại (2 hạt giống ngẫu nhiên khác nhau), không trọng số, trọng số căn bậc 2, có dừng sớm |
| **G3.** Con số 1,000 của bản NetFlow là nhờ **đặc trưng TTL** (Time To Live – số bước nhảy còn lại của gói tin), một "dấu vết" của phòng thí nghiệm UNSW chứ không phải hành vi tấn công. CICFlowMeter không có TTL nên lộ ra độ khó thật | Huấn luyện lại trên bản NetFlow, **có** và **không có** 2 cột MIN_TTL, MAX_TTL |

**Chỉ có 1 ô.** Chạy **sau khi Ô 4 của notebook 05 đã xong** (để không tranh tài nguyên). Không cần GPU (card đồ họa). Mất khoảng 10–20 phút. Kết quả lưu vào `MyDrive/TADP_Q1/results/chan_doan_unsw/`, chạy xong chỉ cần báo Claude.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
from tadp.config import get_config
from tadp.data import make_splits, feature_columns
from tadp.experts import Expert
from tadp.preprocess import Preprocessor
from tadp import metrics as M

B = "/content/drive/MyDrive/TADP_Q1"
OUT = f"{B}/results/chan_doan_unsw"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=OUT, SPLIT_MODE="episode")


def danh_gia(ten, df, cols):
    """Huấn luyện 1 chuyên gia LightGBM (tiền xử lý P1_log) trên phần huấn luyện, chấm điểm trên phần kiểm thử."""
    t0 = time.time()
    sp, _ = make_splits(cfg, {ten: df})
    sp = sp[ten]
    X = df[cols].values
    y = df["Label"].values.astype(np.int64)
    pp = Preprocessor("P1_log").fit_source(X[sp["train"]])
    ex = Expert("lgbm", pp).fit(cfg, X[sp["train"]], y[sp["train"]], X[sp["val"]], y[sp["val"]], seed=11)
    p = ex.proba_from_Z(pp.transform_source(X[sp["test"]]))
    yt, yhat = y[sp["test"]], p >= 0.5
    tp, fp, fn, tn = M._confusion(yt, yhat)
    kq = {"thi_nghiem": ten, "so_dac_trung": len(cols),
          "macro_f1_nguong_0_5": M.macro_f1(yt, yhat),
          "f1_lop_tan_cong": 2 * tp / max(2 * tp + fp + fn, 1),
          "f1_lop_binh_thuong": 2 * tn / max(2 * tn + fn + fp, 1),
          "do_chinh_xac_precision": tp / max(tp + fp, 1), "do_bao_phu_recall": tp / max(tp + fn, 1),
          "macro_f1_nguong_tot_nhat": M.best_threshold_macro_f1(yt, p),
          "auroc": M.safe_auroc(yt, p), "ap": M.safe_ap(yt, p), "ty_le_tan_cong": float(yt.mean())}
    # Độ bao phủ (recall) theo từng loại tấn công
    att = df["Attack"].astype(str).values[sp["test"]]
    theo_loai = (pd.DataFrame({"loai": att[yt == 1], "bat_duoc": yhat[yt == 1]})
                 .groupby("loai")["bat_duoc"].agg(so_luong="size", ty_le_bat_duoc="mean")
                 .sort_values("so_luong", ascending=False))
    # 15 đặc trưng quan trọng nhất (theo tổng mức giảm lỗi – "gain")
    imp = pd.Series(ex.model.feature_importance("gain"), index=cols)
    top = (imp / imp.sum()).sort_values(ascending=False).head(15).round(4)
    print(f"\n=== {ten} ({len(cols)} đặc trưng, {time.time()-t0:.0f} giây) ===")
    print(pd.Series(kq).drop("thi_nghiem").round(4).to_string())
    print("\nĐộ bao phủ theo loại tấn công (trên phần kiểm thử):")
    print(theo_loai.round(3).to_string())
    print("\n15 đặc trưng quan trọng nhất (tỷ lệ đóng góp):")
    print(top.to_string())
    theo_loai.assign(thi_nghiem=ten).to_csv(f"{OUT}/recall_theo_loai__{ten}.csv")
    top.rename("ty_le_dong_gop").to_csv(f"{OUT}/dac_trung_quan_trong__{ten}.csv")
    return kq


def ban_sao_binh_thuong(ten, df, cols):
    """G2: bao nhiêu luồng tấn công có ĐÚNG CÙNG giá trị mọi đặc trưng với ít nhất 1 luồng bình thường."""
    h = pd.util.hash_pandas_object(df[cols].round(6), index=False).values
    y = df["Label"].values
    h_benign = set(h[y == 0])
    co_ban_sao = np.fromiter((v in h_benign for v in h[y == 1]), dtype=bool, count=int((y == 1).sum()))
    att = df["Attack"].astype(str).values[y == 1]
    bang = (pd.DataFrame({"loai": att, "co_ban_sao_binh_thuong": co_ban_sao})
            .groupby("loai")["co_ban_sao_binh_thuong"].agg(so_luong="size", ty_le="mean")
            .sort_values("so_luong", ascending=False))
    tong = float(co_ban_sao.mean())
    print(f"\n=== G2 – {ten}: {tong:.1%} luồng tấn công có bản sao y hệt mang nhãn BÌNH THƯỜNG ===")
    print(bang.round(3).to_string())
    bang.to_csv(f"{OUT}/ban_sao_binh_thuong__{ten}.csv")
    return tong


def thu_lgbm(df, cols):
    """G4: LightGBM có bị "hỏng" vì trọng số lớp quá lớn không? So vài cấu hình LightGBM trên cùng dữ liệu."""
    import math, lightgbm as lgb
    from sklearn.linear_model import LogisticRegression
    sp, _ = make_splits(cfg, {"x": df})
    sp = sp["x"]
    X = df[cols].values
    y = df["Label"].values.astype(np.int64)
    pp = Preprocessor("P1_log").fit_source(X[sp["train"]])
    Ztr, Zva, Zte = (pp.transform_source(X[sp[k]]) for k in ("train", "val", "test"))
    ytr, yva, yte = y[sp["train"]], y[sp["val"]], y[sp["test"]]
    w = (len(ytr) - ytr.sum()) / max(ytr.sum(), 1)
    cau_hinh = [("hien_tai_seed11", w, False, 11), ("hien_tai_seed12", w, False, 12),
                ("khong_trong_so", 1.0, False, 11), ("trong_so_can_bac_2", math.sqrt(w), False, 11),
                ("hien_tai_dung_som", w, True, 11)]
    rows = []
    for ten, spw, dung_som, seed in cau_hinh:
        t0 = time.time()
        clf = lgb.LGBMClassifier(n_estimators=cfg["LGBM_ROUNDS"], learning_rate=0.1, num_leaves=63,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                 scale_pos_weight=spw, random_state=seed, verbose=-1, n_jobs=-1)
        if dung_som:
            clf.fit(Ztr, ytr, eval_set=[(Zva, yva)], callbacks=[lgb.early_stopping(30, verbose=False)])
        else:
            clf.fit(Ztr, ytr)
        s_va = clf.predict(Zva, raw_score=True)
        s_te = clf.predict(Zte, raw_score=True)
        platt = LogisticRegression(C=1e6, max_iter=1000).fit(s_va.reshape(-1, 1), yva)
        p_te = platt.predict_proba(s_te.reshape(-1, 1))[:, 1]
        rows.append({"cau_hinh": ten, "trong_so_lop_tan_cong": round(spw, 1),
                     "so_cay_dung": int(clf.best_iteration_ or cfg["LGBM_ROUNDS"]),
                     "diem_tho_nho_nhat": float(np.min(s_te)), "diem_tho_lon_nhat": float(np.max(s_te)),
                     "he_so_platt": float(platt.coef_[0, 0]),
                     "auroc_kiem_thu": M.safe_auroc(yte, s_te),
                     "macro_f1_nguong_0_5": M.macro_f1(yte, p_te >= 0.5),
                     "macro_f1_nguong_tot_nhat": M.best_threshold_macro_f1(yte, s_te),
                     "giay": round(time.time() - t0)})
    bang = pd.DataFrame(rows).set_index("cau_hinh")
    print("\n=== G4 – Các cấu hình LightGBM trên CIC-UNSW-NB15 ===")
    print(bang.round(4).to_string())
    bang.to_csv(f"{OUT}/g4_cau_hinh_lightgbm.csv")
    return bang


ket_qua = []
# ---------- Bản CICFlowMeter ----------
cic = pd.read_parquet(f"{B}/cache_cic/CIC-UNSW-NB15_1000000.parquet")
cols_cic = feature_columns(cic, "no_id")
thu_lgbm(cic, cols_cic)
kq = danh_gia("CIC-UNSW-NB15", cic, cols_cic)
kq["ty_le_tan_cong_co_ban_sao_binh_thuong"] = ban_sao_binh_thuong("CIC-UNSW-NB15", cic, cols_cic)
ket_qua.append(kq)

# ---------- Bản NetFlow: có và không có TTL ----------
nf = pd.read_parquet(f"{B}/cache/UNSW-NB15_1000000.parquet")
cols_nf = feature_columns(nf, "no_id")
kq = danh_gia("NF-UNSW-NB15_co_TTL", nf, cols_nf)
kq["ty_le_tan_cong_co_ban_sao_binh_thuong"] = ban_sao_binh_thuong("NF-UNSW-NB15_co_TTL", nf, cols_nf)
ket_qua.append(kq)
cols_nf_khong_ttl = [c for c in cols_nf if "TTL" not in c]
print("\nCác cột bị bỏ ở thí nghiệm không TTL:", [c for c in cols_nf if "TTL" in c])
kq = danh_gia("NF-UNSW-NB15_khong_TTL", nf, cols_nf_khong_ttl)
kq["ty_le_tan_cong_co_ban_sao_binh_thuong"] = ban_sao_binh_thuong("NF-UNSW-NB15_khong_TTL", nf, cols_nf_khong_ttl)
ket_qua.append(kq)

tong_hop = pd.DataFrame(ket_qua).set_index("thi_nghiem")
tong_hop.to_csv(f"{OUT}/tong_hop.csv")
print("\n\n================ BẢNG TỔNG HỢP ================")
print(tong_hop.T.round(4).to_string())
print("\nĐã lưu vào", OUT, "– báo cho Claude là xong.")